# Live Coinbase → inline ScreeningPipeline (escalation cascade)

Same agentic operator as `MarketAgentFn` (notebook 07), but driven by a **live Coinbase
WebSocket feed** instead of synthetic ticks. No Kafka, no `flink run` — the WS handler
tumbles ticks into per-product windows and calls `pipeline.screen()` per window.

Funnel: **BandPass (rules) → Z-score (ML) → Claude (LLM, if `ANTHROPIC_API_KEY` set)**.

Prereqs:
- Framework jar built (`mvn -DskipTests package`).
- Venv with `JPype1`, `websockets`, `python-dotenv`, `nest_asyncio` (see `.venv/`).
- `.env` at repo root with `ANTHROPIC_API_KEY=sk-ant-...` to light up the LLM tier.

## 1. Bootstrap — load `.env`, start JVM, build classpath

### Runtime mode

This notebook runs **in-process** by default (`MODE = "inproc"`). The async WebSocket
listener drives the screening operator directly inside this Python process via JPype —
no Flink cluster, no docker.

Other modes:
- `"embedded"` — JPype + in-process MiniCluster. Same caveat as notebook 07: this
  notebook never submits a Flink job.

Edit `MODE` in the next cell to override, or set `AGENTIC_FLINK_MODE` in `.env`.

In [ ]:
# ─── Runtime mode ────────────────────────────────────────────────────────
# Pick where this notebook runs. Edit MODE below to override .env.
#   "inproc"   — JVM inside this Python process (JPype). Fastest, no docker.
#                Works for: 07, 08 (operators are called directly).
#   "session"  — submit jobs to a Flink session cluster via REST.
#                Works for: 09. Set FLINK_URL or FLINK_REST_URL in .env.
#   "embedded" — JPype + in-process Flink MiniCluster (no docker; full job
#                graph but slower than inproc). Works for: 09 (offline test).
MODE = "inproc"          # ← edit to override


In [1]:
import agentic_flink as af
rt = af.bootstrap(mode=MODE)              # MODE set in the cell above
print(rt.info)
import os
key = os.environ.get('ANTHROPIC_API_KEY') or ''


JVM ready — LLM tier: Claude
interpreter: /Users/bengamble/Agentic-Flink/.venv/bin/python


## 2. Build the ScreeningPipeline (same detectors as `MarketAgentFn`)

BandPass on spread (basis points) at PRE-CLASSIFIER, then three Z-score detectors at
CLASSIFIER (spread, total bid volume, total offer volume). LLM tier wakes up only when
rules+ML agree the row is suspicious.

In [2]:
from agentic_flink import jclass
ScreeningPipeline = jclass('org.agentic.flink.screening.ScreeningPipeline')
ScreenItem = jclass('org.agentic.flink.screening.ScreenItem')
BandPass = jclass('org.agentic.flink.screening.BandPassDetector')
ZScore = jclass('org.agentic.flink.screening.ZScoreDetector')
Phase = jclass('org.agentic.flink.screening.Phase')
from java.util import HashMap

# Crypto spreads are tight — BTC-USD typically 0.1–3 bp. Tune the band wider for SOL/altcoins.
SPREAD_BAND_LOW_BP  = 0.0
SPREAD_BAND_HIGH_BP = 25.0
Z_THRESHOLD         = 2.5
WARMUP              = 5

b = (ScreeningPipeline.builder()
       .addDetector(BandPass(SPREAD_BAND_LOW_BP, SPREAD_BAND_HIGH_BP, 0.6))
       .addDetector(ZScore('z-spread', Phase.CLASSIFIER, Z_THRESHOLD, WARMUP, 0.7))
       .addDetector(ZScore.onAttr('totalBidVolume',   Phase.CLASSIFIER, Z_THRESHOLD, WARMUP, 0.5))
       .addDetector(ZScore.onAttr('totalOfferVolume', Phase.CLASSIFIER, Z_THRESHOLD, WARMUP, 0.5))
       .withReviewThreshold(0.6)
       .withBlockThreshold(2.0))
if key.startswith('sk-ant-'):
    b = b.withClaude(key)
pipeline = b.build()
print('pipeline built — LLM tier:', 'Claude' if key.startswith('sk-ant-') else 'disabled')

pipeline built — LLM tier: Claude


## 3. Live Coinbase → windowed features → pipeline

We subscribe to Coinbase's public `ticker` channel (no API key required) and accumulate
ticks into a tumbling window per product. At each window close we synthesise the same
`MarketFeatures` shape the Flink job emits (spread bp + summed bid/offer volume) and feed
it to `pipeline.screen()`.

Stops after `MAX_WINDOWS` windows or when you interrupt the cell.

In [3]:
import asyncio, json, time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import nest_asyncio
import websockets

nest_asyncio.apply()  # allow asyncio in Jupyter

COINBASE_WS      = 'wss://ws-feed.exchange.coinbase.com'
PRODUCTS         = [p.strip() for p in os.environ.get('COINBASE_PRODUCTS', 'BTC-USD,ETH-USD,SOL-USD').split(',') if p.strip()]
WINDOW_SECONDS   = int(os.environ.get('COINBASE_WINDOW_SECONDS', '5'))
MAX_WINDOWS      = int(os.environ.get('COINBASE_MAX_WINDOWS', '24'))  # ~2 min at 5s windows

class TumblingWindow:
    """Accumulates ticker samples for one product across a fixed-duration tumbling window."""
    __slots__ = ('start_ms', 'spread_sum_bp', 'bid_vol_sum', 'offer_vol_sum', 'count')
    def __init__(self, start_ms: int):
        self.start_ms = start_ms
        self.spread_sum_bp = 0.0
        self.bid_vol_sum = 0.0
        self.offer_vol_sum = 0.0
        self.count = 0
    def add(self, spread_bp: float, bid_vol: float, offer_vol: float) -> None:
        self.spread_sum_bp += spread_bp
        self.bid_vol_sum += bid_vol
        self.offer_vol_sum += offer_vol
        self.count += 1
    def features(self, end_ms: int) -> dict:
        n = max(self.count, 1)
        return {
            'windowStart': self.start_ms,
            'windowEnd': end_ms,
            'spread_bp_avg': self.spread_sum_bp / n,
            'totalBidVolume': self.bid_vol_sum,
            'totalOfferVolume': self.offer_vol_sum,
            'samples': self.count,
        }

def _ticker_to_sample(msg: dict):
    """Returns (product, ts_ms, spread_bp, best_bid_size, best_ask_size) or None."""
    try:
        best_bid = float(msg['best_bid']);   best_ask = float(msg['best_ask'])
        if best_bid <= 0 or best_ask <= 0:
            return None
        mid = (best_bid + best_ask) / 2.0
        spread_bp = (best_ask - best_bid) / mid * 10_000.0
        bid_sz = float(msg.get('best_bid_size', 0.0))
        ask_sz = float(msg.get('best_ask_size', 0.0))
        ts_iso = msg.get('time') or ''
        try:
            ts_ms = int(datetime.fromisoformat(ts_iso.replace('Z', '+00:00')).timestamp() * 1000)
        except Exception:
            ts_ms = int(time.time() * 1000)
        return msg['product_id'], ts_ms, spread_bp, bid_sz, ask_sz
    except (KeyError, ValueError, TypeError):
        return None

def _screen_window(product: str, feat: dict, funnel: Counter):
    """Build a Java ScreenItem from the window features and run the pipeline."""
    attrs = HashMap()
    attrs.put('totalBidVolume',   str(feat['totalBidVolume']))
    attrs.put('totalOfferVolume', str(feat['totalOfferVolume']))
    item = ScreenItem(product, float(feat['spread_bp_avg']), 'spread_bp', int(feat['windowEnd']), attrs)
    r = pipeline.screen(item)
    funnel[str(r.decidedBy)] += 1
    phases = ','.join(sorted({str(s.phase()) for s in r.fired})) or '-'
    ts = datetime.fromtimestamp(feat['windowEnd']/1000, tz=timezone.utc).strftime('%H:%M:%S')
    print(
        f"{ts} {product:<8} "
        f"spread={feat['spread_bp_avg']:6.2f}bp "
        f"bid_vol={feat['totalBidVolume']:8.3f} offer_vol={feat['totalOfferVolume']:8.3f} "
        f"n={feat['samples']:<3} | "
        f"{str(r.decidedBy):<5} {str(r.verdict):<6} risk={r.combinedRisk:.2f} [{phases}]"
    )

async def stream_and_screen(products, window_seconds: int, max_windows: int):
    funnel: Counter = Counter()
    windows: dict[str, TumblingWindow] = {}
    closed_count = 0
    sub = json.dumps({'type': 'subscribe', 'product_ids': products,
                      'channels': ['ticker', 'heartbeat']})
    print(f'connecting to {COINBASE_WS}, products={products}, window={window_seconds}s, stop after {max_windows} windows\n')
    async with websockets.connect(COINBASE_WS, ping_interval=20) as ws:
        await ws.send(sub)
        async for raw in ws:
            msg = json.loads(raw)
            if msg.get('type') != 'ticker':
                continue
            sample = _ticker_to_sample(msg)
            if sample is None:
                continue
            product, ts_ms, spread_bp, bid_sz, ask_sz = sample
            bucket_start = (ts_ms // (window_seconds * 1000)) * (window_seconds * 1000)
            w = windows.get(product)
            if w is None:
                windows[product] = TumblingWindow(bucket_start)
                windows[product].add(spread_bp, bid_sz, ask_sz)
                continue
            if bucket_start != w.start_ms:
                # window closed — emit and roll
                _screen_window(product, w.features(w.start_ms + window_seconds * 1000), funnel)
                closed_count += 1
                if closed_count >= max_windows:
                    break
                windows[product] = TumblingWindow(bucket_start)
            windows[product].add(spread_bp, bid_sz, ask_sz)
    print('\nfunnel:', dict(funnel))
    return funnel

funnel = await stream_and_screen(PRODUCTS, WINDOW_SECONDS, MAX_WINDOWS)

connecting to wss://ws-feed.exchange.coinbase.com, products=['BTC-USD', 'ETH-USD', 'SOL-USD'], window=5s, stop after 24 windows

21:52:00 ETH-USD  spread=  0.10bp bid_vol=   4.992 offer_vol=   0.744 n=1   | RULES ALLOW  risk=0.00 [-]
21:52:00 SOL-USD  spread=  1.21bp bid_vol=   3.745 offer_vol=  41.150 n=1   | RULES ALLOW  risk=0.00 [-]
21:52:05 BTC-USD  spread=  0.10bp bid_vol=   0.648 offer_vol=   1.377 n=32  | RULES ALLOW  risk=0.00 [-]
21:52:05 SOL-USD  spread=  1.94bp bid_vol=  25.037 offer_vol= 144.895 n=5   | RULES ALLOW  risk=0.00 [-]
21:52:05 ETH-USD  spread=  0.05bp bid_vol=  56.722 offer_vol=  21.950 n=10  | RULES ALLOW  risk=0.00 [-]
21:52:10 BTC-USD  spread=  0.08bp bid_vol=   0.112 offer_vol=   0.800 n=30  | RULES ALLOW  risk=0.00 [-]
21:52:10 SOL-USD  spread=  1.21bp bid_vol=   4.157 offer_vol=  25.104 n=3   | RULES ALLOW  risk=0.00 [-]
21:52:10 ETH-USD  spread=  0.05bp bid_vol=  13.749 offer_vol=   8.550 n=6   | RULES ALLOW  risk=0.00 [-]
21:52:15 BTC-USD  spread=  0.02

## 4. What just happened

- **Coinbase ticker WS** → per-product samples of best bid/ask + sizes.
- **Tumbling windows** of `WINDOW_SECONDS` collapse samples into `MarketFeatures` rows:
  averaged spread (bp), summed best-bid size, summed best-ask size.
- Each window is fed to the **same `ScreeningPipeline`** the Flink job uses (`MarketAgentFn`):
  - `BandPass` rejects spreads outside `[0, 25]` bp before any ML runs.
  - `ZScore` detectors on spread / bid-volume / offer-volume flag outliers vs. a per-key
    rolling baseline (warmup = 5 windows).
  - When the combined risk crosses the review threshold and Claude is wired in, the LLM
    adjudicates — you'll see `decidedBy=LLM` in the output.

**To see escalations fire:** crypto spreads are usually well-behaved during normal trading,
so the funnel will be mostly `RULES → ALLOW`. Run during high-volatility moments (CPI
release, sudden moves), narrow the band (`SPREAD_BAND_HIGH_BP=2.0`), or add a thinner
altcoin pair to `COINBASE_PRODUCTS` to provoke z-score outliers.

**Mapping back to the Flink job:** every line you see here is what a single subtask of
`MarketAgentFn` (keyed by `instrumentId`) would emit downstream as an `AlertEvent`. The
Java job adds the upstream operators — enrichment, top-N, best-quote⊕trade join,
feature aggregator — but the per-key agentic adjudication is identical.